**Table of contents**<a id='toc0_'></a>    
1. [Pre-requesite](#toc1_)    
1.1. [Python and required packages](#toc1_1_)    
2. [Initialisation](#toc2_)    
3. [Review inconsistent attendance columns (optional)](#toc3_)    
4. [Link public datasets](#toc4_)    
4.1. [Link school performance data](#toc4_1_)    
4.1.1. [Process and add school performance features](#toc4_1_1_)    
4.1.1.1. [Download school performance data and merge them into single file per year](#toc4_1_1_1_)    
4.1.2. [Link school information to student records](#toc4_1_2_)    
4.2. [Link IoD and IMD data](#toc4_2_)    
5. [Derive features](#toc5_)    
5.1. [Prepare the source columns](#toc5_1_)    
5.1.1. [Ethnicity](#toc5_1_1_)    
5.1.2. [Gender](#toc5_1_2_)    
5.1.3. [Language](#toc5_1_3_)    
5.1.4. [School](#toc5_1_4_)    
5.2. [Create the derived features](#toc5_2_)    
5.2.1. [What this step creates](#toc5_2_1_)    
5.2.2. [How the options are used](#toc5_2_2_)    
5.2.3. [Ethnicity features](#toc5_2_3_)    
5.2.4. [Gender feature](#toc5_2_4_)    
5.2.5. [School and language history features](#toc5_2_5_)    
6. [Resolve duplicate and conflicting records](#toc6_)    
7. [Remove columns with identical values](#toc7_)    
8. [Output and next step](#toc8_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=true
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

<div style="background-color:#e8f4f8; border: 2px solid #3399cc; padding: 15px; border-radius: 8px;">

<h1><b>Step-by-step Guidance - Running the NEETML Tool</b></h1>

<span style="display: inline-block; font-size: 20px; color: #2f4f5f; background-color: rgba(255,255,255,0.55); padding: 5px 10px; border-radius: 999px; margin: 2px 0 16px 0;">
  <span style="letter-spacing: 0.4px; text-transform: uppercase; font-size: 20px; font-weight: 700; color: #3399cc;">Author</span>
  <span style="color:#9aa; margin: 0 7px;">•</span>
  <span style="font-weight: 600;">Yanhua Xu</span>
</span>

<div style="font-size: 40px; font-weight: 500; color: #444; border-top: 1px solid #999; padding-top: 8px; margin-top: 6px;">
Step 2 - Feature Engineering
</div>

<!-- # **Step 2 - Feature Engineering** -->
<!-- <h1><b> Step 2 - Feature Engineering </b></h1> -->

This notebook extends the prepared dataset (e.g., merged dataset created in **Step 1**) and turns it into a richer dataset for later modelling.

It includes:  
- reviewing attendance columns whose names changed between years;
- adding school-performance and area-deprivation information;
- creating consistent ethnicity and gender variables;
- summarising each student's language and school history;
- resolving duplicate or conflicting records; and
- removing columns that contain the same values.

⚠️ **Important**

- Unlike Step 1, this step **adds new variables and may change selected values**.
- Existing intermediate files are reused when `overwrite=False`. Set `overwrite=True` only when you intentionally want to rebuild them.
- Missing-value imputation, categorical encoding and scaling are **not** performed here. Those operations should be fitted during model training to avoid data leakage.

**Expected input:** one merged Parquet file from Step 1.  
**Main output:** one feature-engineered Parquet file for the model-training stage.

</div>

# 1. <a id='toc1_'></a>[Pre-requesite](#toc0_)

## 1.1. <a id='toc1_1_'></a>[Python and required packages](#toc0_)

In [ ]:
import sys
from pathlib import Path
import datetime
import pandas as pd
import yaml
from rich import print
import numpy as np

from neetml.feature_engineering import FeatEngineer

import warnings
warnings.filterwarnings("ignore")

# 2. <a id='toc2_'></a>[Initialisation](#toc0_)

This step creates a `FeatEngineer` object and connects it to the merged dataset from Step 1.

The code:

1. finds the configured `merged` folder;
2. selects the merged Parquet file;
3. uses the same filename for all later intermediate outputs; and
4. keeps `overwrite=False` so completed stages can be reused.

> **Check before continuing:** the example assumes there is only one Parquet file in the merged folder. If there are several, select the intended file explicitly.

In [ ]:
overwrite = False

feat_engineer = FeatEngineer(overwrite=overwrite)

# find the path for the merged dataset the produced by the raw data preparation step
from neetml.raw_prep import RawDataCurator
merged_dir = RawDataCurator().get_data_path(keys='merged').Path[0]
merged_path = list(Path(merged_dir).glob('*.parquet'))[0] # Assuming there's only one parquet file in the directory

print(f"Path to the merged dataset: {merged_path}")

output_filename = merged_path.name
feat_engineer.set_output_filename(output_filename)

# 3. <a id='toc3_'></a>[Review inconsistent attendance columns (optional)](#toc0_)

Attendance extracts collected in different years do not always use the same column names. If equivalent columns remain separate, one cohort may appear to have missing data simply because its information is stored under another name.

This optional step follows three stages:

1. **Find possible matches.** `find_comp_pairs()` looks for columns with similar names and complementary coverage.
2. **Review the mapping.** Suggested matches are saved in `col_map.yaml`. A reviewed version can be saved as `col_map_curated.yaml`.
3. **Apply the mapping and tidy known fields.** The code combines accepted columns, rebuilds selected attendance measures, splits combined school identifiers and saves a curated dataset.

::: {.callout-warning}

**Manual review is important**

A similar name or value range does not prove that two fields have the same definition. On a first run, the generated mapping is used when a reviewed mapping is not available, so inspect `col_map.yaml` and the coverage plots before relying on the curated output.

<details>
<summary style="background-color:#f0f8ff; padding:8px; border:1px solid #ccc; border-radius:5px; cursor:pointer; font-weight:bold; color:#003366;">
Known attendance fields requiring additional review
</summary>

- `attendance_educational_count` could refer to an educational visit or a broader attendance count, so it is not mapped automatically.
- Holiday, traveller, illness and exceptional-circumstance fields have changed names or value ranges between extracts. Their mappings should be checked against the source metadata.
- Several fields appear in only one cohort, including `attendance_late`, `attendance_registration_closed` and `attendance_not_covered`. They are dropped when no reliable equivalent can be identified.

</details>

:::

**Additional processing in the code**

- fills selected missing attendance totals when their component counts agree;
- splits `laestab` into local-authority and establishment fields;
- recalculates possible sessions from attended and absence counts;
- recalculates percentage fields from their corresponding count fields; and
- produces attendance and census coverage plots to help identify remaining schema drift.

**Output:** a curated Parquet file in the `2_merged_curated` folder, together with the YAML mapping and coverage plots. If the curated file already exists and `overwrite=False`, it is loaded instead of being rebuilt.

In [ ]:
from neetml.utils.misc import find_comp_pairs, apply_col_map, plot_col_coverage

# create the dir for the curated dataset to be saved
curated_dir = merged_path.parent.parent / (merged_path.parent.name + '_curated')
curated_dir.mkdir(exist_ok=True)
# curated_path = curated_dir / merged_path.with_name(merged_path.stem + '_curated.parquet').name
curated_path = curated_dir / output_filename

if curated_path.exists() and not overwrite:
    df = pd.read_parquet(curated_path) # or set input_data_path to curated_path
    print(f'The curated dataset has been loaded from {curated_path}.')
else:
    print(f"Curated dataset does not exist at {curated_path}. Processing the merged dataset to create the curated dataset...")
    df = pd.read_parquet(merged_path)

    # 'attendance_absence_count' is new variable since 2022-2023_Y7, so fill the missing values of 'attendance_absence_count' with the sum of 'attendance_authorised_count' and 'attendance_unauthorised_count' if they are the same for the non-missing values, to keep the most complete data for 'attendance_absence_count'
    if not (df[['attendance_authorised_count', 'attendance_unauthorised_count']].sum(axis=1, skipna=True) == df['attendance_absence_count']).eq(False).any():
        # if 'attendance_absence_count' is the same as the sum of 'attendance_authorised_count' and 'attendance_unauthorised_count', then fill the missing values
        df['attendance_absence_count'].fillna(df['attendance_unauthorised_absence'] + df['attendance_authorised_absence'], inplace=True)
        df['attendance_absence_count'].fillna(df['attendance_authorised'] + df['attendance_unauthorised'], inplace=True)

    # 'attendance_new_possible' only exists in 2022-2023_attendance_Y11, it seems contains the updated 
    # It seems 'attendance_authorised' (abscence count) + 'attendance_unauthorised' (abscence count) + 'attendance_sessions_attended' = 'attendance_new_possible'
    if not (df[['attendance_authorised', 'attendance_unauthorised', 'attendance_sessions_attended']].sum(axis=1, skipna=True) == df['attendance_new_possible']).eq(False).any():
        df['attendance_new_possible'].fillna(df['attendance_possible_sessions'], inplace=True) # 'attendance_new_possible' maybe the updated version of 'attendance_possible_sessions' in 2022-2023_Y11, so fill the missing values with 'attendance_possible_sessions' to keep the most complete data for 'attendance_possible_sessions'
        df.drop(columns=['attendance_possible_sessions'], inplace=True)
        df.rename(columns={'attendance_new_possible': 'attendance_possible_sessions'}, inplace=True)

    
    print(f"\n Scanning data for complementary, name-similar & dtype-check column pairs...")

    cols = [
        col for col in df.columns
        if col != 'stud_id'
        and not col.startswith(('_', 'ks2', 'ks4'))
    ]

    _, minor_to_major_map = find_comp_pairs(df=df, cols=cols, thres=0.8) 
    
    save_path = curated_dir / "col_map.yaml"
    with open(save_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(
            minor_to_major_map,
            f,
            sort_keys=True,
            allow_unicode=True,
            default_flow_style=False
        )
    
    curated_yaml_path = save_path.stem + "_curated.yaml"
    
    if (curated_dir / curated_yaml_path).exists():
        col_map = yaml.load((curated_dir / curated_yaml_path).open(), Loader=yaml.FullLoader)
    else:
        col_map = yaml.load((curated_dir / save_path.name).open(), Loader=yaml.FullLoader)
    
    df_curated = apply_col_map(df, col_map, drop_from_col=True, copy=True, verbose=True)
    
    print('Done. The complementary, name-similar & dtype-check column pairs have been mapped.')

    # Some columns need to be recalculated or split
    # 'attendance_laestab' contains la code and establishment code, which can be split into two separate columns 'attendance_la' and filled the missing value of 'attendance_estab_no' 
    df_curated['attendance_la'] = df_curated['attendance_laestab'].str[:3]
    df_curated['attendance_estab_no'].fillna(df_curated['attendance_laestab'].str[3:], inplace=True)

    # do the same for ks2 and ks4 laestab columns
    df_curated['ks2_la'] = df_curated['ks2_laestab'].str[:3]
    df_curated['ks4_la'] = df_curated['ks4_laestab'].str[:3]
    
    # 'attendance_possible_sessions' may not be accurated. It seems it should be number of attended sessions + unauthorised and authorised absences, so recalculate it to keep the most complete data for 'attendance_possible_sessions'
    df_curated.drop(columns=['attendance_possible_sessions'], inplace=True)
    df_curated['attendance_possible_sessions'] = df_curated['attendance_count'] + df_curated['attendance_unauthorised_count'] + df_curated['attendance_authorised_count']
        
    # drop some columns that only exists in one cohort but cannot be mapped to any other columns in other cohorts
    df_curated.drop(columns=['attendance_late', 'attendance_registration_closed', 'attendance_not_covered', 'attendance_hash', 'attendance_pupil_not_yet_on_roll', 'attendance_illness_confirmed_coronavirus', 'attendance_non_compulsory_school_age', 'attendance_dash'], inplace=True, errors='ignore')
    
    # also drop 'attendance_possible_sessions_less_covid_absence' as it only exists in three cohorts
    df_curated.drop(columns=['attendance_possible_sessions_less_covid_absence'], inplace=True, errors='ignore')
    
    # percentage columns only exist in new data, so fill the missing values with the percentage calculated from the count columns
    pct_cols = [col for col in df_curated.columns if '%' in col]
    for pct_col in pct_cols:
        count_col = pct_col.replace('%', 'count')
        if count_col in df_curated.columns:
            den = df_curated["attendance_possible_sessions"]
            num = df_curated[count_col]
            mask = den.notna() & (den > 0) & num.notna()
            
            df_curated[pct_col] = np.nan
            df_curated.loc[mask, pct_col] = num[mask] / den[mask]
        else:
            print(f"Warning: The count column {count_col} for {pct_col} is not found. The percentage column will be filled with NaN.")
            df_curated[pct_col] = pd.NA
    
    for col in ['attendance', 'census']:
        plot_col_coverage(
            df_curated,
            prefix=col,
            figsize=(25, 15),
            verbose=True,
            save_excel=True,
            output_file=str(curated_dir / f'{col}_cols_presence.jpg')
        )
    
    df_curated.to_parquet(curated_path, index=False)
    
    df = df_curated.copy()

Before linking external data, check the paths used by `FeatEngineer`.

The table below shows the input, external-data, intermediate-output and final-output locations. 

<!-- If the input path is not the curated dataset you intend to use, set it with `feat_engineer.set_input_data_path(curated_path)` and display the paths again. -->

In [ ]:
feat_engineer.get_data_path(keys='all')

In [ ]:
# feat_engineer.set_input_data_path(curated_path)

# feat_engineer.get_data_path(keys='all')

# 4. <a id='toc4_'></a>[Link public datasets](#toc0_)

This section adds:

- information about the schools connected to each record; and
- neighbourhood deprivation information linked through postcode.

These are left joins: the original student records are retained, while unmatched external fields remain missing.

## 4.1. <a id='toc4_1_'></a>[Link school performance data](#toc0_)

School context can help describe the environment in which a student was educated. Because `attendance`, `suspension`, `KS2` and `KS4` records may refer to different schools or years, each source is linked separately.

School linking has two parts:

1. prepare one standard reference table from the annual school-performance downloads; and
2. join the relevant school and year to each group of student records.

The download/build step is optional when the reference table already exists.

### 4.1.1. <a id='toc4_1_1_'></a>[Process and add school performance features](#toc0_)

#### 4.1.1.1. <a id='toc4_1_1_1_'></a>[Download school performance data and merge them into single file per year](#toc0_)

Historical school data can be downloaded from [Compare School Performance](https://www.compare-school-performance.service.gov.uk/download-data).

The code cell is commented out because downloading and combining the files only needs to be done when the external reference data is first created or refreshed.

- Use `la = 380` to download Bradford schools only.
- Use `la = 0` to include all schools in England. This is useful when some students attend school outside Bradford.
- `build_school_perf_data()` combines the annual downloads into one reference table.
- No school-performance data is expected for the 2019-2020 academic year.
- Percentage fields are kept as percentage values; for example, `85.5` represents 85.5%.

**Output:** a combined school reference file in the configured external-data folder.

In [ ]:
# # la = 380 # Bradford
# la = 0 # All of England (as some children studied outside of bradford)
# fmt = 'xlsx'
# current_year = datetime.date.today().year

# fetch_kwargs={
#     "la_code": la,
#     "academic_start_year": 2013,
#     "academic_end_year": current_year,
#     "file_format": fmt,
# }

# feat_engineer.download_school_data(**fetch_kwargs)

# df_school = feat_engineer.build_school_perf_data()

### 4.1.2. <a id='toc4_1_2_'></a>[Link school information to student records](#toc0_)

The school data is linked in three stages so each join can use the correct school identifier and year.

| Stage | Student records used | Year used for matching | Saved output |
|---|---|---|---|
| 1 | Attendance and suspension/exclusion | `_acad_year` | `link_perf_1.parquet` |
| 2 | KS2 | `ks2_examyear_re` | `link_perf_2.parquet` |
| 3 | KS4 | `_cohort_y11_ay` | `link_perf_3.parquet` |

Attendance, suspension and KS4 use the previous year's school data by default; KS2 uses the assessment year. This prevents later school information from being attached to an earlier student record.

Each stage uses the output from the previous stage. If the saved file already exists and `overwrite=False`, it is loaded instead of rebuilding the join.

**Result:** `df` contains the original student columns plus the matched school features. A missing school feature means that no suitable reference record was found for that school and year.

In [ ]:
df_att_susp = feat_engineer.link_school_perf_data(input_data=df, school_ref=["att", "susp"], output_name="link_perf_1.parquet") # by default the input_year_col = '_acad_year'
df_ks2 = feat_engineer.link_school_perf_data(input_data=df_att_susp, school_ref=["ks2"], input_year_col='ks2_examyear_re', output_name="link_perf_2.parquet")
df = feat_engineer.link_school_perf_data(input_data=df_ks2, school_ref=['ks4'], input_year_col='_cohort_y11_ay', output_name="link_perf_3.parquet")

del df_att_susp, df_ks2

## 4.2. <a id='toc4_2_'></a>[Link IoD and IMD data](#toc0_)

The Index of Deprivation (IoD) and Index of Multiple Deprivation (IMD) describe the area in which a student lives. They are neighbourhood measures, not direct measures of the individual student or family.

Earlier pre-Year-11 records may contain an NCCIS status but no NCCIS postcode. To improve postcode coverage, the first code cell:

1. finds each student's postcode in the Year 11 pre-spring record;
2. copies it to earlier records where the postcode is missing; and
3. leaves existing postcodes unchanged.

This assumes that the Year 11 postcode is a reasonable proxy for the earlier address. Keep this assumption in mind when interpreting deprivation features for students who may have moved.

The next cell links the postcode to the 2019 IoD reference data and saves `link_perf_iod.parquet`.

**Result:** `df` contains the linked deprivation fields. Records with a missing or unmatched postcode will keep missing deprivation values.

In [ ]:
# Optional pre-processing steps
# For records before Year 11, only NCCIS code was appended, so NCCIS postcode is missing for those records.
# To keep NCCIS postcode as complete as possible, fill the missing postcode values in pre-Year-11 records
# with the corresponding NCCIS postcode recorded at Year 11.

if 'nccis_postcode' in df.columns and 'nccis_code' in df.columns:
    # get each person's Year 11 postcode
    y11_postcode = (
        df.loc[(df["_year_group"] == '11') & (df["_phase"] == "pre-spring"), ["stud_id", "nccis_postcode"]]
        .dropna(subset=["nccis_postcode"])
        .drop_duplicates()
        .set_index("stud_id")["nccis_postcode"]
    )

    # map Year 11 postcode back to all records
    df["_y11_postcode"] = df["stud_id"].map(y11_postcode)

    # fill missing postcode ONLY for records before Year 11
    mask = (df["_year_group"].astype(int) < 11) & (df["nccis_postcode"].isna())

    df.loc[mask, "nccis_postcode"] = df.loc[mask, "_y11_postcode"]
    
    df.drop(columns="_y11_postcode", inplace=True)

In [ ]:
df = feat_engineer.add_iod(input_data=df, col_pd='nccis_postcode', output_name="link_perf_iod.parquet")

# 5. <a id='toc5_'></a>[Derive features](#toc0_)

The linked dataset still contains the same concepts in several source columns. This section combines those observations and creates clearer variables that describe the student's characteristics and history.

The process has two parts:

1. prepare consistent source columns; and
2. derive the final ethnicity, gender, language and school-history features.

## 5.1. <a id='toc5_1_'></a>[Prepare the source columns](#toc0_)

The same information can appear in several datasets and in different formats. These short preparation steps create one consistent input for each feature group before the main derivation function is called.

### 5.1.1. <a id='toc5_1_1_'></a>[Ethnicity](#toc0_)

The `census_ethnicity` column contains a mixture of (extended and main) ethnicity codes (e.g. `WBRI, AIND`) and about 20\% main/sub category labels (e.g. `Pakistani, Asian Or Asian British`), all of which need to be standardised before use.

Process steps:
1. curate the `census_ethnicity` with `suspPermExcl_ethnicity`
2. map all curated codes to DfE main category labels
3. replace following labels that cannot be found in DfE main category labels:
    `White British And Irish` -> `White`,
    `Mixed` -> `Mixed/Dual Background`,
    `Other Ethnic` -> `Any Other Ethnic Group`
    `Not On Jan Census` -> `Unknown`

Reference: [DfE census ethnicity codes](https://www.gov.uk/guidance/alternative-provision-ap-census/codes).

In [ ]:
# from neetml.utils.constants import DER_COL_PREFIX

# the current census_ethnicity column contains a mixture of ethnicity codes (e.g. `WBRI, AIND`) and labels (e.g. `White other, Pakistani, Asian Or Asian British`)
# where it seems the labels are from the approved extended categories and main categories, so we use suspPermExcl_ethnicity to curate the census_ethnicity column

df["_ethnicity_raw"] = df["suspPermExcl_ethnicity"].astype("object").fillna(df["census_ethnicity"].astype("object"))    

df = df.drop(columns=['suspPermExcl_ethnicity', 'census_ethnicity']).drop_duplicates()

df["_ethnicity_raw"].replace(
    {
        'White British And Irish': 'White',
        'Mixed': 'Mixed/Dual Background',
        'Other Ethnic': 'Any Other Ethnic Group',
        'Not On Jan Census': "Unknown"
    }, inplace=True)

### 5.1.2. <a id='toc5_1_2_'></a>[Gender](#toc0_)

Gender/sex is available in several source tables and may use full labels or short codes.

The code identifies the relevant columns, while excluding fields whose names contain "sex" or "type" but describe a different concept. It then standardises:

- `Female` to `F`;
- `Male` to `M`; and
- `Unknown` to a missing value.

The list printed below shows exactly which source columns will be used to create the final gender feature.

In [ ]:
gender_cols = [col for col in df.columns if any(x in col.lower() for x in ["gender", "sex"]) and col != 'ks2_gpsexp' and 'type' not in col]
df[gender_cols] = df[gender_cols].replace({"Female": "F", "Male": "M", "Unknown": pd.NA})

print(gender_cols)

### 5.1.3. <a id='toc5_1_3_'></a>[Language](#toc0_)

Language is available from census and suspension/exclusion data.

This step creates `_language_raw` by using `census_language` first and filling missing values from `suspPermExcl_language`. The two original columns are then removed.

The working column is temporary. It will be used to build the student's language history and main recorded language.

In [ ]:
language_cols = [col for col in df.columns if any(x in col.lower() for x in ["language"])]
print(language_cols)

df["_language_raw"] = df["census_language"].astype("object").fillna(df["suspPermExcl_language"].astype("object"))    

df = df.drop(columns=['census_language', 'suspPermExcl_language'], errors='ignore').drop_duplicates()

### 5.1.4. <a id='toc5_1_4_'></a>[School](#toc0_)

School information is spread across several columns, so this preparation uses URN, postcode, school type and gender type together.

The three code cells below:

1. remove `nccis_establishment_type`, which has too few populated values to be useful;
2. convert Year Group and phase into a clear time order, where pre-spring comes before post-spring;
3. map KS2/KS4 school-type codes to readable categories; and
4. collect the school columns that will be passed to the derivation function.

Using several school details makes the later school-change indicator less sensitive to one missing or inconsistent identifier.

In [ ]:
df = df.drop(columns=['nccis_establishment_type'], errors='ignore') # only two rows have non-noll values in 'nccis_establishment_type'

df['_year_group'] = df['_year_group'].astype("Int64")
df['_phase_num'] = df['_phase'].astype(object).fillna(0).replace('pre-spring', 1).replace('post-spring', 2)

In [ ]:
from neetml.utils.code_mappings import ks_nftype_map

# map ks2/ks4 school type codes to categories
nftype_cols = [col for col in df.columns if 'nftype' in col]
for c in nftype_cols:
    new_cols = c.replace('nftype', 'school_type')
    df[new_cols] = df[c].astype("Int64").map(ks_nftype_map)

In [ ]:
urn_cols = [col for col in df.columns if 'urn' in col and not col.startswith('ks2') and 'ac' not in col]
pcd_cols = [col for col in df.columns if 'postcode' in col and 'ks2' not in col and 'nccis' not in col]
gender_type_cols = [col for col in df.columns if 'gender_type' in col and 'ks2' not in col]

exclude_cols = {"ks4_nftype", "ks4_new_type", "ks4_newer_type", "nccis_establishment_type"} # not considered them for now

type_cols = [
    col for col in df.columns
    if "school_type" in col
    and "ks2" not in col
    and col not in exclude_cols
]

## 5.2. <a id='toc5_2_'></a>[Create the derived features](#toc0_)

The prepared source columns are now passed to `derive_features()`. This creates the new variables in one consistent step and saves an intermediate dataset in the configured `4_derived` folder.

### 5.2.1. <a id='toc5_2_1_'></a>[What this step creates](#toc0_)

| Feature group | Main result |
|---|---|
| Ethnicity | standard DfE ethnicity levels, a curated value and conflict indicators |
| Gender | one curated gender value per student where the sources agree |
| School | school histories, main school characteristics and a school-change flag |
| Language | language history, main language and number of recorded languages |

<!-- All history features are ordered by Year Group and phase, so each row summarises information available up to that point. -->

### 5.2.2. <a id='toc5_2_2_'></a>[How the options are used](#toc0_)

- `check_level="main_cat"` checks ethnicity conflicts at the broadest DfE category level.
- `time_cols=["_year_group", "_phase_num"]` defines the order used for school and language histories.
- `features=("hist", "main", "state_chg")` requests school history, the main value and a change indicator.
- `state_threshold=2` requires at least two school details to change before a school-state change is flagged.
- Language requests `hist`, `main` and `n_uniq`.

After derivation, array values are converted to tuples so duplicate rows can be compared reliably. The temporary source columns and original gender columns are then removed.

### 5.2.3. <a id='toc5_2_3_'></a>[Ethnicity features](#toc0_)

One raw ethnicity value can be represented at several levels of detail. For example, a detailed code can be mapped to an extended category, a sub-category and a broad main category.

| New variable | Meaning |
|---|---|
| `der_ethnicity_ext_code` | detailed DfE ethnicity code |
| `der_ethnicity_ext_cat` | detailed approved category |
| `der_ethnicity_sub_cat` | broader sub-category |
| `der_ethnicity_main_code` | DfE main code |
| `der_ethnicity_main_cat` | broad main category |
| `der_ethnicity_curated` | final value at the selected check level |
| `der_ethnicity_conflict_n` | number of different substantive values found |
| `der_ethnicity_conflict_flag` | whether conflicting values were found |

With `check_level="main_cat"`, a conflict is raised only when the same student has more than one broad ethnicity category. Non-substantive entries such as “Refused”, “Unknown” and “Not obtained” are not counted as conflicting values.

If a conflict is found, `der_ethnicity_curated` is left missing rather than choosing one value automatically.

### 5.2.4. <a id='toc5_2_4_'></a>[Gender feature](#toc0_)

The pipeline compares all gender/sex source columns selected earlier.

- If the sources contain one consistent value, it becomes `der_gender`.
- If both `F` and `M` are found for the same student, `der_gender` is left missing rather than selecting one arbitrarily.

The log output reports the number of affected students. In the saved run below, 32 students had conflicting gender values.

### 5.2.5. <a id='toc5_2_5_'></a>[School and language history features](#toc0_)

For each student, the pipeline orders records by Year Group and phase and summarises the information observed up to the current row.

**School features**

- `hist`: the school values observed so far;
- `main`: the most frequently observed value; and
- `state_chg`: whether the school state changed at the current point.

A school change is flagged only when at least two school details change.

**Language features**

- `hist`: the languages observed so far;
- `main`: the main recorded language; and
- `n_uniq`: the number of different recorded languages.

These variables describe the recorded history. A change may reflect a real change or a difference between source systems, so unusual cases should still be reviewed.

In [ ]:
df_new = feat_engineer.derive_features(
    input_data=df,
    input_path=curated_path,
    derive_ethnicity=True,
    derive_gender=True,
    derive_school=True,
    derive_language=True,
    ethnicity_kwargs={
        "ethnic_col": "_ethnicity_raw",
        "check_level": "main_cat",
        # "time_cols": ['_year_group', '_phase_num'], 
    },
    gender_kwargs={
        "gender_cols": gender_cols,
        # "time_cols": ['_year_group', '_phase_num'], 
    },
    school_kwargs={
        "urn_cols": urn_cols,
        "pcd_cols": pcd_cols,
        "type_cols": type_cols,
        "gender_type_cols": gender_type_cols,
        "time_cols": ['_year_group', '_phase_num'], 
        "clean_type": True,
        "features": ("hist", "main", "state_chg"),
        "state_threshold": 2
    },
    language_kwargs={
        "language_col": "_language_raw",
        "time_cols": ['_year_group', '_phase_num'], 
        "features": ("hist", "main", "n_uniq"),
    }
)

df = df_new.copy()

array_cols = [
    col for col in df.columns
    if df[col].map(lambda x: isinstance(x, np.ndarray)).any()
]

if array_cols:

    print(array_cols)

    for col in array_cols:
        df[col] = df[col].map(
            lambda x: tuple(x.tolist()) if isinstance(x, np.ndarray) else x
        )

df = df.drop(columns=["_ethnicity_raw", "_language_raw"] + gender_cols, errors='ignore').drop_duplicates()

# 6. <a id='toc6_'></a>[Resolve duplicate and conflicting records](#toc0_)

After merging several sources, one intended student record may still appear in several rows. A single rule would not work for every data type: two attendance rows should usually be added together, while two census rows may require choosing the most relevant enrolment record.

The output is grouped by:

`stud_id + cohort + Year Group + academic year + phase`

The resolver first identifies affected students and then applies a rule suited to each data category.

| Category | How conflicts are resolved |
|---|---|
| Census | choose the record with the highest enrolment-status priority |
| KS2 | choose a base record, then fill its missing result fields |
| KS4 | choose the most complete/base record, then fill missing result fields |
| Attendance | sum counts and recalculate percentages |
| Suspension/exclusion | keep the latest event details and count the events |

<details>
<summary style="background-color:#f0f8ff; padding:8px; border:1px solid #ccc; border-radius:5px; cursor:pointer; font-weight:bold; color:#003366;">
How each rule works
</summary>

**Census**

The preferred enrolment status is `M`, followed by `C`, then `S`, then any other or missing value.

**KS2 and KS4**

The most complete and plausible row is selected as the base. Missing attainment or background fields may be filled from another duplicate row, but an existing base value is not overwritten. School IDs, candidate IDs and year fields are not mixed between records.

**Attendance**

Count and session fields are summed. Percentage fields are recalculated from the combined counts instead of averaging percentages from rows with different numbers of sessions.

**Suspension and permanent exclusion**

These rows can represent valid separate events. The latest event details are retained and an event-count feature records how many events occurred in the group.

</details>

Before consolidation, the detailed ethnicity-level columns are removed; the curated ethnicity and conflict indicators are retained.

With `use_conflict_ids=True`, the pipeline saves a conflict report and processes only the students who need consolidation.

**Output:** the consolidated dataset is saved in the configured `5_aggregated` folder.

In [ ]:
df = df.drop(columns=['der_ethnicity_main_cat', 'der_ethnicity_main_code', 'der_ethnicity_sub_cat', 'der_ethnicity_ext_cat', 'der_ethnicity_ext_code'], errors='ignore').drop_duplicates()

df_new = feat_engineer.resolve_conflicts(
  input_data=df,
  overwrite=False,
  group_keys=['stud_id', '_cohort_y11_ay', '_year_group', '_acad_year', '_phase_num'],
  use_conflict_ids=True,
)

# 7. <a id='toc7_'></a>[Remove columns with identical values](#toc0_)

The final dataset may contain columns that are exact copies of one another after linking and derivation.

The first code cell hashes each column, keeps the first column for every repeated hash and saves the reduced dataset in the main processed-data folder. The next cells display the result and perform an additional pairwise check for columns that still contain identical values.

> **Review point:** confirm that the retained column has the clearest meaning before using the final dataset. Two columns can have the same values in the current data even when their names describe different concepts.

In [ ]:
col_hashes = df_new.apply(lambda s: pd.util.hash_pandas_object(s, index=False).sum())

keep_cols = col_hashes[~col_hashes.duplicated()].index

df_new_dedup = df_new[keep_cols]

df_new_dedup.to_parquet(merged_dir.parent / output_filename, index=False)

In [ ]:
df_new_dedup

In [ ]:
# double check if there are any columns that have the same values across all rows

same_cols = []

cols = sorted(df_new.columns)

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if df_new[cols[i]].equals(df_new[cols[j]]):
            same_cols.append((cols[i], cols[j]))

same_cols

# 8. <a id='toc8_'></a>[Output and next step](#toc0_)

At the end of this notebook, the processed-data folder contains the feature-engineered dataset saved by the previous step. The intermediate folders also retain the externally linked, derived and consolidated versions so the workflow can be traced or restarted.

Before moving on, check that:

- the expected cohort and Year Groups are present;
- the number of students is reasonable;
- external school and deprivation fields have acceptable coverage;
- the derived histories follow the correct time order; and
- no unexpected identical columns remain.

The dataset is now ready for **Step 3 - Model Training**, where missing-value handling, encoding, scaling and model fitting should be performed.